# 5 — Support Ticket Router
> Every inbound ticket is automatically classified, assigned to the right team, and handed back with a ready-to-send first reply — so your agents spend their time resolving issues, not routing them.

*Run all cells. Swap the input in the final code cell with your own data.*

In [ ]:
%pip install -q langchain-openai langchain-core pydantic python-dotenv
import os
os.environ['OPENAI_API_KEY'] = 'sk-...'  # replace

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field
from langchain_core.messages import SystemMessage
from langchain_openai import ChatOpenAI


# --- Data models ---

class TicketClassification(BaseModel):
    ticket_type: Literal["billing", "technical", "account", "feature_request", "other"]
    urgency: Literal["critical", "high", "medium", "low"]
    team: Literal["billing", "engineering", "account_management", "product", "general_support"]
    confidence: float = Field(ge=0.0, le=1.0, description="Classifier confidence 0-1")
    reasoning: str = Field(description="One sentence explaining the routing decision")


class DraftReply(BaseModel):
    subject: str
    body: str
    internal_note: str = Field(description="Note for the support agent, not sent to customer")
    escalate: bool = Field(description="True if this needs a human to review before sending")


# --- Prompts ---

CLASSIFIER_PROMPT = SystemMessage(
    """You are a customer support ticket classifier.

Given a support ticket, classify it with:
- ticket_type: billing | technical | account | feature_request | other
- urgency:
    critical = service down, data loss, security issue
    high     = major feature broken, billing dispute
    medium   = degraded performance, billing question
    low      = general question, feature request
- team: the team best equipped to handle it
    billing          → billing/technical/account issues involving payment
    engineering      → bugs, outages, data issues
    account_management → account changes, upgrades, cancellations
    product          → feature requests, feedback
    general_support  → everything else
- confidence: your certainty 0.0–1.0
- reasoning: one sentence explaining your routing decision"""
)

DRAFT_PROMPTS = {
    "billing": SystemMessage(
        """You draft first-response emails for the billing support team.
Be empathetic, acknowledge the issue, and set a clear expectation for resolution time (1-2 business days for billing disputes).
Always include: acknowledgment, next steps, and a contact path if urgent.
Set escalate=True for any dispute over $500 or involving a subscription cancellation."""
    ),
    "engineering": SystemMessage(
        """You draft first-response emails for the engineering/technical support team.
Acknowledge the issue, ask for relevant details (OS, browser, steps to reproduce) if not provided.
Set escalate=True for any outage, data loss, or security concern."""
    ),
    "account_management": SystemMessage(
        """You draft first-response emails for the account management team.
Be warm and professional. For cancellation requests, acknowledge and offer a retention path.
Set escalate=True for enterprise accounts or churn risk."""
    ),
    "product": SystemMessage(
        """You draft first-response emails for the product team.
Thank the customer for feedback, confirm it's been logged, and set expectations (no commit to timelines).
escalate=False unless the feature request is blocking their use of the product."""
    ),
    "general_support": SystemMessage(
        """You draft first-response emails for general support.
Be helpful and concise. Aim to resolve in one reply where possible.
Set escalate=True only if the issue requires account access or billing changes."""
    ),
}

DRAFT_CONTEXT = """
You are replying to this customer support ticket:

Subject: {subject}
From: {customer_name} <{customer_email}>

---
{body}
---

Classification: {ticket_type} / {urgency} urgency → routed to {team}
"""


# --- Pipeline ---

def create_classifier():
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    return CLASSIFIER_PROMPT | llm.with_structured_output(TicketClassification)


def create_drafter(team: str):
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)
    system = DRAFT_PROMPTS.get(team, DRAFT_PROMPTS["general_support"])
    return system | llm.with_structured_output(DraftReply)


def run(ticket: dict) -> dict:
    """
    ticket: {subject, body, customer_name, customer_email}
    returns: {classification, draft}
    """
    classifier = create_classifier()
    classification: TicketClassification = classifier.invoke(
        f"Subject: {ticket['subject']}\n\n{ticket['body']}"
    )

    context = DRAFT_CONTEXT.format(
        subject=ticket["subject"],
        customer_name=ticket["customer_name"],
        customer_email=ticket["customer_email"],
        body=ticket["body"],
        ticket_type=classification.ticket_type,
        urgency=classification.urgency,
        team=classification.team,
    )

    drafter = create_drafter(classification.team)
    draft: DraftReply = drafter.invoke(context)

    return {
        "classification": classification.model_dump(),
        "draft": draft.model_dump(),
    }


print("Ready.")

## The scenario

A customer has written in to say they can no longer export their data and suspects their account permissions changed after a recent plan upgrade. The ticket is a mix of technical frustration and account anxiety — the kind that slips through the cracks when routing is done manually. The agent will classify it, decide which team owns it, and hand back a ready-to-send reply with an internal note for the agent handling it.

In [ ]:
ticket = {
    "subject": "Export stopped working after upgrading to Pro",
    "body": (
        "Hi, I upgraded our account to Pro last Tuesday and now the CSV export "
        "button on the Reports page is greyed out for everyone on my team. "
        "We use this export every week for our finance team — it's blocking us right now. "
        "I checked the permissions settings and everything looks right on our end. "
        "Could this be a bug with the upgrade, or did something change with Pro plan exports?"
    ),
    "customer_name": "Daniel Osei",
    "customer_email": "d.osei@meridianops.com",
}

result = run(ticket)
clf = result["classification"]
draft = result["draft"]

print(f"{'='*60}")
print(f"TICKET: {ticket['subject']}")
print(f"FROM:   {ticket['customer_name']} <{ticket['customer_email']}>")
print(f"{'='*60}")

print("\n[ROUTING DECISION]")
print(f"  Type:        {clf['ticket_type']}")
print(f"  Urgency:     {clf['urgency']}")
print(f"  Assigned to: {clf['team']}")
print(f"  Confidence:  {clf['confidence']:.0%}")
print(f"  Reason:      {clf['reasoning']}")

print("\n[DRAFT REPLY]")
print(f"  Subject:   {draft['subject']}")
print(f"  Escalate:  {'YES — hold for human review' if draft['escalate'] else 'No — ready to send'}")
print("\n  Body:\n")
for line in draft["body"].split("\n"):
    print(f"    {line}")
print(f"\n  [Internal note]: {draft['internal_note']}")

## Use your own data

Replace the `ticket` dict above with:
- `subject` and `body` — the raw text from an inbound ticket
- `customer_name` and `customer_email` — the sender's details

The agent will return the assigned team, urgency level, a ready-to-send reply, and an internal note — along with an escalation flag if the ticket needs a human to review it before it goes out.